In [5]:
from pyspark.sql import SparkSession
session=SparkSession.builder.appName("RDD Basics").getOrCreate()
sc=session.sparkContext

In [6]:
#how to make it rdd-parallelize
numbers = [10, 20, 30, 40, 50]

rdd = sc.parallelize(numbers)

print(rdd)

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297


In [7]:
rdd.collect()

[10, 20, 30, 40, 50]

In [8]:
rdd.getNumPartitions()

2

In [10]:
rdd=sc.parallelize(numbers,3)
rdd.getNumPartitions()

3

In [11]:
rdd=sc.parallelize([10,20,30,40,50],3)
rdd.glom().collect()
#glom converts ech partition to a list

[[10], [20, 30], [40, 50]]

**Lazy evaluation**- doesnt do anything unless it has action

RDD operations: two categories
1. Transformations
These create a new RDD.
Examples:
map
filter
flatMap
distinct
union
intersection
groupByKey
reduceByKey
2. Actions
These return a result to the driver.
Examples:
collect
count
first
take
reduce

In [12]:
rdd=sc.parallelize ([10,20,30,40,50],3)

mapped_rdd = rdd.map(lambda x: x * 2)
filtered_rdd = mapped_rdd.filter(lambda x: x > 5)

print("Nothing executed yet")

Nothing executed yet


In [15]:
print(filtered_rdd.collect())
#function happens or transformation happens only when action is performed
print(mapped_rdd.collect())

[20, 40, 60, 80, 100]
[20, 40, 60, 80, 100]


In [17]:
rdd = sc.parallelize([1, 2, 3, 4, 5])

squared_rdd = rdd.map(lambda x:x*x)

squared_rdd.collect()

[1, 4, 9, 16, 25]

In [18]:
rdd=sc.parallelize(["Vaishli","PC","Computer",'Priki'])
upper_rd=rdd.map(lambda x:x.upper())
upper_rd.collect()

['VAISHLI', 'PC', 'COMPUTER', 'PRIKI']

In [19]:
rdd=sc.parallelize(["apple",'banana','strawberry'])
long_word=rdd.filter(lambda x:len(x)>5)
long_word.collect()

['banana', 'strawberry']

flatmap-can catch multiple elements

In [20]:
lines=sc.parallelize([
    "python spark hadoop",
    "spark sql pyspark"
])
words=lines.flatMap(lambda x:x.split(" "))
words.collect()

['python', 'spark', 'hadoop', 'spark', 'sql', 'pyspark']

gives list inside list

In [22]:
lines=sc.parallelize(['hello world',"pyspark rdd"])
result=lines.map(lambda x: x.split(" "))
result.collect()

[['hello', 'world'], ['pyspark', 'rdd']]

In [23]:
lines=sc.parallelize(['hello world',"pyspark rdd"])
result=lines.flatMap(lambda x: x.split(" "))
result.collect()#-> gives directly in a single list

['hello', 'world', 'pyspark', 'rdd']

In [25]:
rdd=sc.parallelize([10,20,30,20,34,506,96,79])
rdd.distinct().collect()

[10, 20, 30, 34, 506, 96, 79]

In [27]:
rdd1=sc.parallelize([10,20,30,40,50])
rdd2=sc.parallelize([60,70,80])
rdd1.union(rdd2).collect()

[10, 20, 30, 40, 50, 60, 70, 80]

In [28]:
rdd1=sc.parallelize([10,20,30,40,50])
rdd2=sc.parallelize([60,70,80,30,40,50])
rdd1.intersection(rdd2).collect()

[40, 30, 50]

In [29]:
rdd1=sc.parallelize([10,20,30,40,50])
rdd2=sc.parallelize([60,70,80,30,40,50])
rdd1.subtract(rdd2).collect()

[20, 10]

**Actions**

In [30]:
rdd=sc.parallelize([10,20,30])
rdd.collect()

[10, 20, 30]

In [31]:
rdd=sc.parallelize([10,20,30])
rdd.count()

3

In [32]:
rdd=sc.parallelize([10,20,30])
rdd.first()

10

In [33]:
rdd=sc.parallelize([10,20,30,40,50])
rdd.take(3)

[10, 20, 30]

In [34]:
rdd=sc.parallelize([10,20,30,40])
rdd.reduce(lambda a,b:a+b)

100

In [37]:
data=[("IT",70000),
 ("AI",9000),
 ("HR",890009),
 ("AI",90000),
 ("IT",60000)]
rdd=sc.parallelize(data)
rdd.collect()

[('IT', 70000), ('AI', 9000), ('HR', 890009), ('AI', 90000), ('IT', 60000)]

In [39]:
rdd.reduceByKey(lambda a,b:a+b).collect()

[('IT', 130000), ('AI', 99000), ('HR', 890009)]

In [42]:
grouped=rdd.groupByKey()
[(k,list(v)) for k,v in grouped.collect()]

[('IT', [70000, 60000]), ('AI', [9000, 90000]), ('HR', [890009])]

mapValues- only works on values

In [43]:
rdd=sc.parallelize([
    ("Rahul",86),
    ("Sneha",76),
    ("Arjun",78)
])
rdd.mapValues(lambda x:x+2).collect()

[('Rahul', 88), ('Sneha', 78), ('Arjun', 80)]

In [45]:
rdd=sc.parallelize([10,20,30,4])
rdd.sortBy(lambda x:x).collect()


[4, 10, 20, 30]

In [46]:
lines = sc.parallelize([
"python spark python",
"spark hadoop spark",
"python hadoop"
])
word_counts=(
    lines.flatMap(lambda x:x.split(" "))
    .map(lambda x:(x,1))
    .reduceByKey(lambda a,b:a+b)
)
word_counts.collect()


[('python', 3), ('hadoop', 2), ('spark', 3)]

In [47]:
employees = sc.parallelize([
("Rahul", 70000),
("Sneha", 60000),
("Arjun", 75000),
("Priya", 80000),
("Karan", 50000)
])
high_sal=employees.filter(lambda x:x[1]>65000)
high_sal.collect()

[('Rahul', 70000), ('Arjun', 75000), ('Priya', 80000)]